In [2]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_parquet("../Data/raw/all_race_control.parquet")
df.head()

,year,round,session,session_folder,event,country,location,event_date,Time,Category,Message,Status,Flag,Scope,Sector,RacingNumber,Lap
0,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,2021-03-28 14:45:06,Other,RISK OF RAIN FOR F1 RACE IS 0%,NaN,NaN,NaN,NaN,NaN,1.0
1,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,2021-03-28 14:57:06,Drs,DRS DISABLED,DISABLED,NaN,NaN,NaN,NaN,1.0
2,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,2021-03-28 14:57:40,Flag,YELLOW IN TRACK SECTOR 12,NaN,YELLOW,Sector,12.0,NaN,1.0
3,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,2021-03-28 14:57:41,Flag,CLEAR IN TRACK SECTOR 12,NaN,CLEAR,Sector,12.0,NaN,1.0
4,2021,1,Race,01_Race,Bahrain Grand Prix,Bahrain,Sakhir,2021-03-28,2021-03-28 15:03:30,Other,START ABORTED - EXTRA FORMATION LAP,NaN,NaN,NaN,NaN,NaN,1.0


In [21]:
# Shape of the df
print("Dataset shape:", df.shape)

Dataset shape: (10191, 17)


10,191 rows and 17 columns.

In [23]:
# List of all the columns
print(df.columns.tolist())

['year', 'round', 'session', 'session_folder', 'event', 'country', 'location', 'event_date', 'Time', 'Category', 'Message', 'Status', 'Flag', 'Scope', 'Sector', 'RacingNumber', 'Lap']


In [19]:
# Data type in each column
df.dtypes

year                       int64
round                      int64
session                      str
session_folder               str
event                        str
country                      str
location                     str
event_date        datetime64[ns]
Time              datetime64[ns]
Category                     str
Message                      str
Status                       str
Flag                         str
Scope                        str
Sector                   float64
RacingNumber                 str
Lap                      float64
dtype: object

In [18]:
# Missing values
df.isna().sum().sort_values(ascending=False)

Status            9591
Sector            8388
RacingNumber      6931
Scope             4769
Flag              4769
Lap                  2
year                 0
round                0
session              0
Time                 0
event_date           0
location             0
country              0
event                0
session_folder       0
Category             0
Message              0
dtype: int64

The large number of missing values in Status, Flag, Scope, Sector and RacingNumber is likely related to the different types of race-control messages.

The two missing Lap values will also need to be checked.

In [17]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 4


Four exact duplicate rows.

Duplicate records will be checked to find our whether they are genuine duplicate entries or repeated race-control events. 

The 2 records with the missing lap numbers will also be checked.

In [24]:
# Duplicate records
df[df.duplicated(keep=False)].sort_values(["year", "event", "Time"])

,year,round,session,session_folder,event,country,location,event_date,Time,Category,Message,Status,Flag,Scope,Sector,RacingNumber,Lap
5055,2023,11,Race,11_Race,Hungarian Grand Prix,Hungary,Budapest,2023-07-23,2023-07-23 14:40:22,Flag,WAVED BLUE FLAG FOR CAR 3 (RIC) TIMED AT 16:40:21,NaN,BLUE,Driver,NaN,3,70.0
5065,2023,11,Race,11_Race,Hungarian Grand Prix,Hungary,Budapest,2023-07-23,2023-07-23 14:40:22,Flag,WAVED BLUE FLAG FOR CAR 3 (RIC) TIMED AT 16:40:21,NaN,BLUE,Driver,NaN,3,70.0
5056,2023,11,Race,11_Race,Hungarian Grand Prix,Hungary,Budapest,2023-07-23,2023-07-23 14:41:14,Flag,CHEQUERED FLAG,NaN,CHEQUERED,Track,NaN,NaN,70.0
5066,2023,11,Race,11_Race,Hungarian Grand Prix,Hungary,Budapest,2023-07-23,2023-07-23 14:41:14,Flag,CHEQUERED FLAG,NaN,CHEQUERED,Track,NaN,NaN,70.0
7509,2024,20,Race,20_Race,Mexico City Grand Prix,Mexico,Mexico City,2024-10-27,2024-10-27 19:08:01,Other,PINK HEAD PADDING MATERIAL MUST BE USED,NaN,NaN,NaN,NaN,NaN,1.0
7510,2024,20,Race,20_Race,Mexico City Grand Prix,Mexico,Mexico City,2024-10-27,2024-10-27 19:08:01,Other,PINK HEAD PADDING MATERIAL MUST BE USED,NaN,NaN,NaN,NaN,NaN,1.0
7390,2024,19,Race,19_Race,United States Grand Prix,United States,Austin,2024-10-20,2024-10-20 18:07:00,Other,PINK HEAD PADDING MATERIAL MUST BE USED,NaN,NaN,NaN,NaN,NaN,1.0
7391,2024,19,Race,19_Race,United States Grand Prix,United States,Austin,2024-10-20,2024-10-20 18:07:00,Other,PINK HEAD PADDING MATERIAL MUST BE USED,NaN,NaN,NaN,NaN,NaN,1.0


We found 4 exact duplicate rows.

Since every column matches exactly, we will remove the extra copies.

In [28]:
# Remove exact duplicate records
df = df.drop_duplicates()

print("Dataset shape after removing duplicates:", df.shape)
print("Remaining duplicate rows:", df.duplicated().sum())

Dataset shape after removing duplicates: (10187, 17)
Remaining duplicate rows: 0


Duplicates have been removed.

In [26]:
# Records with missing lap
df[df["Lap"].isna()][["year", "event", "Time", "Category", "Message", "Flag", "Status"]]

,year,event,Time,Category,Message,Flag,Status
1240,2021,Russian Grand Prix,2021-09-26 11:01:05,Other,PINK HEAD PADDING MATERIAL MUST BE USED,NaN,NaN
6221,2024,Miami Grand Prix,2024-05-05 19:08:46,Other,PINK HEAD PADDING MATERIAL MUST BE USED,NaN,NaN


Both missing values are associated with the message *PINK HEAD PADDING MATERIAL MUST BE USED*.

In [27]:
# Race control event categories.
df["Category"].value_counts()

Category
Flag         5422
Other        4029
Drs           320
SafetyCar     280
CarEvent      140
Name: count, dtype: int64

The 280 SafetyCar records are important because safety-car periods can significantly change pit-stop strategy.

The recorded safety care messages, statuses and associated ingo will be inspected in order to understand how these events are represented in the dataset.

In [32]:
# Number of safety cars
safety_car_df = df[df["Category"] == "SafetyCar"]

print("Number of Safety Car records:", len(safety_car_df))

Number of Safety Car records: 280


In [34]:
# Safety car statuses
display(safety_car_df["Status"].value_counts(dropna=False))

Status
DEPLOYED                148
IN THIS LAP              68
ENDING                   55
THROUGH THE PIT LANE      9
Name: count, dtype: int64

- *DEPLOYED* appears 148 times.
- *IN THIS LAP* appears 68 times.
- *ENDING* appears 55 times.
- *THROUGH* THE PIT LANE appears 9 times.

In [36]:
# Safety car messages.
safety_car_df[["year", "event", "Time", "Message", "Status", "Lap"]].head(20)

,year,event,Time,Message,Status,Lap
7,2021,Bahrain Grand Prix,2021-03-28 15:08:12,SAFETY CAR DEPLOYED,DEPLOYED,1.0
9,2021,Bahrain Grand Prix,2021-03-28 15:12:15,SAFETY CAR IN THIS LAP,IN THIS LAP,3.0
15,2021,Bahrain Grand Prix,2021-03-28 15:15:33,VIRTUAL SAFETY CAR DEPLOYED,DEPLOYED,4.0
16,2021,Bahrain Grand Prix,2021-03-28 15:16:16,VIRTUAL SAFETY CAR ENDING,ENDING,5.0
80,2021,Emilia Romagna Grand Prix,2021-04-18 13:04:56,SAFETY CAR DEPLOYED,DEPLOYED,2.0
83,2021,Emilia Romagna Grand Prix,2021-04-18 13:15:06,SAFETY CAR IN THIS LAP,IN THIS LAP,6.0
161,2021,Emilia Romagna Grand Prix,2021-04-18 13:54:44,SAFETY CAR DEPLOYED,DEPLOYED,32.0
169,2021,Emilia Romagna Grand Prix,2021-04-18 14:22:42,SAFETY CAR THROUGH THE PIT LANE,THROUGH THE PIT LANE,34.0
208,2021,Portuguese Grand Prix,2021-05-02 14:05:04,SAFETY CAR DEPLOYED,DEPLOYED,2.0
209,2021,Portuguese Grand Prix,2021-05-02 14:05:09,SAFETY CAR THROUGH THE PIT LANE,THROUGH THE PIT LANE,2.0


The dataset include both safety cars and virtual safety cars, so we must distinguish between them because they have different effects on pit-stop strategy.

DEPLOYED alone doesn't tell us whether it's an SC or VSC. The actual message will need to be inspected.

In [37]:
# Distinguishing safety car and virtual safety car events.
safety_car_df["Message"].value_counts(dropna=False)

Message
SAFETY CAR DEPLOYED                81
SAFETY CAR IN THIS LAP             68
VIRTUAL SAFETY CAR DEPLOYED        67
VIRTUAL SAFETY CAR ENDING          55
SAFETY CAR THROUGH THE PIT LANE     9
Name: count, dtype: int64

- Safety Car (SC): Physical safety car on track. It can bunch up the field and reduce the relative time cost of a pit stop.
- Virtual Safety Car (VSC): Drivers follow a controlled speed delta, without a physical safety car bunching them up in the same way.

We will keep the original Message and Status values. 


Flag messages provide information about track conditions, including yellow flags, red flags, blue flags and clear-track notifications. 

We will examine the flag values and messages to understand which events are recorded.

In [41]:
# Number of flag records
flag_df = df[df["Category"] == "Flag"]

print("Number of flag records:", len(flag_df))

Number of flag records: 5420


In [42]:
# Flag values
display(flag_df["Flag"].value_counts(dropna=False))

Flag
BLUE               2995
CLEAR              1066
YELLOW              454
DOUBLE YELLOW       430
GREEN               218
BLACK AND WHITE     124
CHEQUERED           110
RED                  23
Name: count, dtype: int64

In [43]:
# Most common flag messages.
display(flag_df["Message"].value_counts().head(20))

Message
GREEN LIGHT - PIT EXIT OPEN        218
TRACK CLEAR                        147
CHEQUERED FLAG                     110
CLEAR IN TRACK SECTOR 2             79
CLEAR IN TRACK SECTOR 9             68
CLEAR IN TRACK SECTOR 3             65
CLEAR IN TRACK SECTOR 7             61
CLEAR IN TRACK SECTOR 6             59
CLEAR IN TRACK SECTOR 5             59
CLEAR IN TRACK SECTOR 4             58
CLEAR IN TRACK SECTOR 10            54
CLEAR IN TRACK SECTOR 8             49
CLEAR IN TRACK SECTOR 14            47
CLEAR IN TRACK SECTOR 13            44
YELLOW IN TRACK SECTOR 2            43
CLEAR IN TRACK SECTOR 16            43
DOUBLE YELLOW IN TRACK SECTOR 2     40
YELLOW IN TRACK SECTOR 6            39
CLEAR IN TRACK SECTOR 11            39
CLEAR IN TRACK SECTOR 12            37
Name: count, dtype: int64

No changes needed.

Car events can help explain incidents that trigger neutralisations, while DRS messages provide information about when the system is enabled or disabled.

In [44]:
# Investigating car events and DRS messeges.
for category in ["CarEvent", "Drs"]:
    category_df = df[df["Category"] == category]

    print(f"\n{category}: {len(category_df)} records")

    print("\nMessages:")
    display(category_df["Message"].value_counts().head(20))

    print("\nStatuses:")
    display(category_df["Status"].value_counts(dropna=False))


CarEvent: 140 records

Messages:


Message
CAR 44 (HAM) OFF TRACK AND CONTINUED AT TURN 12    3
CAR 47 (MSC) MISSED THE APEX OF TURN 4             2
CAR 4 (NOR) MISSED THE APEX OF TURN 2              2
CAR 47 (MSC) SPUN AND CONTINUED AT TURN 4          2
CAR 16 (LEC) MISSED THE APEX OF TURN 2             2
CAR 4 (NOR) OFF TRACK AND CONTINUED AT TURN 5      2
CAR 10 (GAS) OFF TRACK AND CONTINUED AT TURN 13    2
CAR 20 (MAG) OFF TRACK AND CONTINUED AT TURN 2     2
CAR 6 (LAT) MISSED THE APEX OF TURN 14             2
CAR 22 (TSU) MISSED THE APEX OF TURN 14            2
CAR 5 (VET) SPUN AND CONTINUED AT TURN 1           2
CAR 20 (MAG) OFF TRACK AND CONTINUED AT TURN 4     2
CAR 63 (RUS) MISSED THE APEX OF TURN 10            2
CAR 24 (ZHO) MISSED THE APEX OF TURN 10            2
CAR 23 (ALB) MISSED THE APEX OF TURN 10            2
CAR 10 (GAS) MISSED THE APEX OF TURN 14            2
CAR 2 (SAR) SPUN AND CONTINUED AT TURN 6           2
CAR 55 (SAI) OFF TRACK AND CONTINUED AT TURN 7     1
CAR 55 (SAI) OFF TRACK AND CONTINUED A


Statuses:


Status
NaN    140
Name: count, dtype: int64


Drs: 320 records

Messages:


Message
DRS ENABLED     196
DRS DISABLED    124
Name: count, dtype: int64


Statuses:


Status
ENABLED     196
DISABLED    124
Name: count, dtype: int64

The messages describe incidents such as:
- Drivers going off track and rejoining.
- Drivers spinning but continuing.
- Drivers missing corners.

All 140 Status values are missing which is not necessarily a problem because these messages describe incidents directly in the Message column, rather than using a separate status field.


The Other category contains general race-control announcements that do not fall into the other defined categories. We will inspect the most frequent messages to identify potentially useful information and any inconsistencies.

In [47]:
# Number of other records
other_df = df[df["Category"] == "Other"]

print("Number of Other records:", len(other_df))

Number of Other records: 4027


In [46]:
# Most common messages.
display(other_df["Message"].value_counts().head(30))

Message
PIT EXIT CLOSED                              104
DRS DISABLED IN ZONE 1                        76
DRS DISABLED IN ZONE 2                        66
DRS ENABLED IN ZONE 1                         61
DRS ENABLED IN ZONE 2                         59
RISK OF RAIN FOR F1 RACE IS 0%                49
PINK HEAD PADDING MATERIAL MUST BE USED       41
DRS DISABLED IN ZONE 3                        25
DRS ENABLED IN ZONE 3                         17
LOW GRIP CONDITIONS                           17
TRACK SURFACE SLIPPERY IN TRACK SECTOR 5      15
TRACK SURFACE SLIPPERY IN TRACK SECTOR 17     15
TRACK SURFACE SLIPPERY IN TRACK SECTOR 2      15
ALL PASS HOLDERS MAY ACCESS THE PIT LANE      14
TRACK SURFACE SLIPPERY IN TRACK SECTOR 4      13
TRACK SURFACE SLIPPERY IN TRACK SECTOR 10     13
PIT LANE ENTRY OPEN                           12
TRACK SURFACE SLIPPERY IN TRACK SECTOR 11     12
TRACK SURFACE SLIPPERY IN TRACK SECTOR 3      11
TRACK SURFACE SLIPPERY IN TRACK SECTOR 6      11
TRACK SURFAC

DRS zone messages appear under Other, whereas general DRS enabled/disabled messages appear under Drs.

We also have slightly different rain-message formats, such as 0% and 0 %. We will preserve them and extract structured information later during feature engineering.

In [51]:
# Shape and duplicates
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

Dataset shape: (10187, 17)
Duplicate rows: 0


In [52]:
# Event category count
display(df["Category"].value_counts())

Category
Flag         5420
Other        4027
Drs           320
SafetyCar     280
CarEvent      140
Name: count, dtype: int64

In [53]:
# Remaining missing values
display(df.isna().sum()[df.isna().sum() > 0])

Status          9587
Flag            4767
Scope           4767
Sector          8384
RacingNumber    6928
Lap                2
dtype: int64

In [58]:
# Save the cleaned dataset
output_path = "../Data/processed/all_race_control_cleaned.parquet"

df.to_parquet(output_path, index=False)

# Verify the saved file
df_check = pd.read_parquet(output_path)

print("\nSaved successfully:", output_path)
print("Saved dataset shape:", df_check.shape)


Saved successfully: ../Data/processed/all_race_control_cleaned.parquet
Saved dataset shape: (10187, 17)
